# Lab 2 · Classification: EOR screening
**Slides:** Module 06 and 07 (classification metrics, ROC)  ·  **Time:** 75 min

455 field projects worldwide, each with the EOR method that was applied: CO₂, hydrocarbon or nitrogen gas injection,
polymer flooding, in-situ combustion or steam. From six reservoir properties we predict the method, then compare the
machine-learning screen with the classic **Taber screening rules**.

**You will:** handle imbalanced classes · compare classifiers · read confusion matrices · understand micro, macro and weighted averages · screen a new field.

In [ ]:
# --- setup: works locally and in Google Colab ---
import os, sys
if "google.colab" in sys.modules and not os.path.exists("aiml"):
    print("Upload the course kit zip to Colab, then unzip it:  !unzip -q aiml_course_kit.zip && %cd aiml_course_kit")
sys.path.insert(0, os.path.abspath(".."))
sys.path.insert(0, os.path.abspath("."))
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": .3, "axes.spines.top": False, "axes.spines.right": False})
DATA = "../data" if os.path.exists("../data") else "data"

from aiml.data import load_eor, EOR_FEATURES, EOR_CLASSES
from aiml.models import classifiers
from aiml.evaluate import compare, confusion, classification_metrics, importance
from aiml.generators import taber_screen, TABER
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
df = load_eor(log_viscosity=True)
print(df.eor_method.value_counts())

## 1 · Inputs: raw or log?
Viscosity runs from 0.1 to 5,000,000 cP. Trees do not care about scale; distance-based models (KNN, SVC, MLP) do.

In [ ]:
F_RAW = EOR_FEATURES
F_LOG = ["log_permeability", "depth_ft", "api_gravity", "log_viscosity", "temperature_F", "oil_saturation_pct"]
y = df.eor_method
tr, te = train_test_split(df.index, test_size=0.2, random_state=42, stratify=y)
for name in ["KNN", "SVC", "Random forest"]:
    m = classifiers()[name]
    a = m.fit(df.loc[tr, F_RAW], y[tr]).score(df.loc[te, F_RAW], y[te]); b = m.fit(df.loc[tr, F_LOG], y[tr]).score(df.loc[te, F_LOG], y[te])
    print(f"{name:<14} raw inputs {a:.3f}   log inputs {b:.3f}")

## 2 · Compare classifiers
With imbalanced classes, accuracy hides the small classes; sort by **macro F1** (every class counts the same).

In [ ]:
X_train, X_test, y_train, y_test = df.loc[tr, F_LOG], df.loc[te, F_LOG], y[tr], y[te]
res = compare(classifiers(), X_train, y_train, X_test, y_test, task="classification")
res[["model", "acc_train", "accuracy_test", "F1_macro_test", "F1_micro_test", "F1_weighted_test", "ROC_AUC_ovr_test"]]

**Micro** averaging pools all predictions (equals accuracy for single-label problems). **Macro** averages the per-class
scores, so Nitrogen (7 projects) counts as much as Steam (197). **Weighted** averages by class size. The course
figures `Micro_Average.JPG`, `Macro_Average.JPG` and `Weighted_Average.JPG` show the same three numbers.

In [ ]:
rf = classifiers()["Random forest"].fit(X_train, y_train); p = rf.predict(X_test)
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
confusion(y_test, p, EOR_CLASSES, ax=ax[0], title="Random forest · test counts")
confusion(y_test, p, EOR_CLASSES, ax=ax[1], title="row-normalized (recall per class)", normalize=True)
plt.tight_layout(); plt.show()
from sklearn.metrics import classification_report; print(classification_report(y_test, p, zero_division=0))

### ✏️ Your turn 1 · cross-validated comparison
One 20% test set holds only ~2 Nitrogen projects, so its scores are noisy. Compare *Random forest*, *Gradient boosting*,
*SVC* and *Neural network (MLP)* with **stratified 5-fold cross-validation** (`scoring="f1_macro"`) on all data. Report mean ± std.

In [ ]:
cv = StratifiedKFold(5, shuffle=True, random_state=0)

## 3 · Machine learning vs the expert rules
`taber_screen` applies simplified Taber, Martin & Seright (1997) criteria, the rules an engineer would use.

In [ ]:
pd.DataFrame(TABER).T

In [ ]:
t = taber_screen(df.loc[te])
print("Taber rules  accuracy", round((t == y_test).mean(), 3), " macro F1", round(classification_metrics(y_test, t)["F1_macro"], 3))
print("Random forest accuracy", round((p == y_test).mean(), 3), " macro F1", round(classification_metrics(y_test, p)["F1_macro"], 3))
fig, ax = plt.subplots(figsize=(7, 5.5)); confusion(y_test, t, EOR_CLASSES, ax=ax, title="Taber rules · test"); plt.show()

The rules know nothing about this database, yet get most projects right; the model learns the operators' real
choices (for example, which light-oil reservoirs got CO₂ rather than hydrocarbon gas). Rules are transparent and work
outside the data; the model is more accurate inside it. Use both.

## 4 · Explain and screen a new field

In [ ]:
imp = importance(rf, X_test, y_test); plt.title("permutation importance (accuracy drop)"); plt.show()

In [ ]:
new = pd.DataFrame([{"permeability_md": 100, "depth_ft": 8000, "api_gravity": 30, "viscosity_cp": 10, "temperature_F": 150, "oil_saturation_pct": 50}])
new_log = new.assign(log_permeability=np.log10(new.permeability_md), log_viscosity=np.log10(new.viscosity_cp))[F_LOG]
print("model:", rf.predict(new_log)[0], "  rules:", taber_screen(new)[0])
pd.Series(rf.predict_proba(new_log)[0], rf.classes_).sort_values(ascending=False).round(3)

### ✏️ Your turn 2
A heavy-oil field: permeability 1500 mD, depth 1200 ft, 12 °API, 3000 cP, 95 °F, 70 % oil saturation.
What do the model and the rules recommend, and how confident is the model? Then make the same field 6000 ft deep. What changes?

In [ ]:
heavy = pd.DataFrame([{"permeability_md": 1500, "depth_ft": 1200, "api_gravity": 12, "viscosity_cp": 3000, "temperature_F": 95, "oil_saturation_pct": 70}])

### ✏️ Your turn 3 · class weights
Retrain the random forest with `class_weight="balanced"` and compare recall for *Nitrogen*, *Combustion* and *Polymer*.

## Key ideas
* Log-transform inputs that span decades before distance-based models.
* With imbalanced classes report macro F1 and the confusion matrix, not just accuracy.
* Cross-validate when the test set is small.
* Expert rules are a strong, explainable baseline; ML should beat them and agree with them where it matters.

In [ ]:
# RandomForestClassifier(n_estimators=300, class_weight='balanced', random_state=0)